# SuperSimpleNet — Augmentation ablation runner

Runs **every** augmentation config in `configs/*.json` (or a chosen subset), one
Drive folder per config. **Stop/resume-safe**: on restart it skips any config
that already finished (a `DONE.txt` marker), so you never lose completed work.

Workflow:
1. Run **Setup** once (mount Drive, clone your branch, install deps).
2. Run **Copy dataset to local** once (fast I/O; re-points `DATA_PATH` locally).
3. (SUP mode only) Run **Dataset prep** once to seed anomalies into the train set.
4. Run the **Grid runner** — re-run it any time; it resumes where it left off.
5. Run **Summary** to collect all metrics into one CSV on Drive.


## 1) Setup — mount Drive, clone branch, install dependencies

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ========================= CONFIGURE ME =========================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
# IMPORTANT: your augmentation branch (must carry aug_config.py / augment_ssn.py / configs/).
BRANCH_NAME  = 'augmentation'
REPO_PATH    = '/content/SuperSimpleNet'

DATA_PATH    = '/content/drive/MyDrive/Tesi/datasets/MVTec'
# Base Drive folder for the WHOLE ablation grid (one sub-folder per config is created here).
ABLATION_BASE = '/content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation'

CATEGORY = 'Nero_SEMISUPERVISED_dustValidationAndTrain'
MODE     = 'sup'          # 'sup' (supervised/mixed) or 'unsup'

# Shared training hyperparameters (identical across configs so only augmentation varies)
EPOCHS = 300
BATCH  = 4
SEED   = 42
# ===============================================================

if not os.path.exists(REPO_PATH):
    print(f">>> Cloning branch '{BRANCH_NAME}'...")
    !git clone -b {BRANCH_NAME} {GIT_REPO_URL} {REPO_PATH}
else:
    print(f">>> Repo present. Checking out '{BRANCH_NAME}' and pulling...")
    os.chdir(REPO_PATH)
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

os.chdir(REPO_PATH)
os.makedirs(ABLATION_BASE, exist_ok=True)
print("Working dir:", os.getcwd())
print("Ablation base:", ABLATION_BASE)

!pip install tqdm anomalib==0.7
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install wandb optuna
# ONNX deps (needed only if RUN_ONNX_EXPORT=True in the grid runner)
!pip install onnx onnxscript onnxruntime onnxconverter-common
# CRITICAL: keep NumPy < 2.0 -- anomalib 0.7 pulls imgaug, which uses np.sctypes
# (removed in NumPy 2.0). Pin it LAST so no earlier install can bump it back to 2.x.
# train.py runs as a subprocess and reads this on-disk numpy, so no restart is needed.
!pip install "numpy<2.0"


Mounted at /content/drive
>>> Cloning branch 'augmentation'...
Cloning into '/content/SuperSimpleNet'...
remote: Enumerating objects: 452, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 452 (delta 181), reused 177 (delta 126), pack-reused 201 (from 2)
Receiving objects: 100% (452/452), 242.52 KiB | 7.13 MiB/s, done.
Resolving deltas: 100% (248/248), done.
Working dir: /content/SuperSimpleNet
Ablation base: /content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.0/158.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━

## 1b) Copy dataset to local storage (fast I/O)

Reading the test/train images over Google Drive (FUSE) is the main reason eval is
slow, and in `sup` mode the loader is stuck at 1 worker. This copies the category
to local `/content` storage once and re-points `DATA_PATH` there for the rest of
the notebook. Idempotent: skipped if the local copy already exists (per session).


In [2]:
import os, shutil, time

# Source on Drive (whatever Setup pointed DATA_PATH at) -> local destination.
DRIVE_DATA = DATA_PATH
LOCAL_DATA = '/content/mvtec_local'
src = os.path.join(DRIVE_DATA, CATEGORY)
dst = os.path.join(LOCAL_DATA, CATEGORY)

if not os.path.isdir(src):
    raise FileNotFoundError(f"Source category not found on Drive: {src}")

if os.path.isdir(dst) and os.listdir(dst):
    print(f">>> Local copy already present, skipping copy: {dst}")
else:
    os.makedirs(LOCAL_DATA, exist_ok=True)
    print(f">>> Copying {src}\n           -> {dst} ...")
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f">>> Copy done in {time.time() - t0:.1f}s")

# Point the pipeline at the local copy for all downstream cells.
DATA_PATH = LOCAL_DATA
print(">>> DATA_PATH now ->", DATA_PATH)


>>> Copying /content/drive/MyDrive/Tesi/datasets/MVTec/Nero_SEMISUPERVISED_dustValidationAndTrain
           -> /content/mvtec_local/Nero_SEMISUPERVISED_dustValidationAndTrain ...
>>> Copy done in 858.2s
>>> DATA_PATH now -> /content/mvtec_local


## 2) Dataset prep (SUP mode only — run once)

Seeds a few anomalous samples (with masks) into the train set, exactly like the
main notebook. Idempotent: safe to re-run. Skip entirely for `unsup` mode.

In [3]:
import os, glob, shutil

NUM_ANOMALIES_PER_DEFECT = 2

if MODE == 'sup':
    dataset_root = os.path.join(DATA_PATH, CATEGORY)
    test_dir  = os.path.join(dataset_root, 'test')
    gt_root   = os.path.join(dataset_root, 'ground_truth')
    train_dir = os.path.join(dataset_root, 'train')

    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Cannot find test folder: {test_dir}")

    defect_types = [d for d in os.listdir(test_dir)
                    if os.path.isdir(os.path.join(test_dir, d)) and d != 'good']
    print(f">>> SUP prep for '{CATEGORY}', defects: {defect_types}")

    for defect in defect_types:
        defect_test_dir = os.path.join(test_dir, defect)
        defect_gt_dir   = os.path.join(gt_root, defect)
        target_train_defect_dir = os.path.join(train_dir, defect)
        target_train_gt_dir     = os.path.join(train_dir, 'ground_truth', defect)
        os.makedirs(target_train_defect_dir, exist_ok=True)
        os.makedirs(target_train_gt_dir, exist_ok=True)

        images = sorted(glob.glob(os.path.join(defect_test_dir, "*.png")))
        for img_path in images[:NUM_ANOMALIES_PER_DEFECT]:
            stem = os.path.splitext(os.path.basename(img_path))[0]
            dest_img = os.path.join(target_train_defect_dir, os.path.basename(img_path))
            if not os.path.exists(dest_img):
                shutil.copy(img_path, dest_img)
            for mask_path in (os.path.join(defect_gt_dir, f"{stem}_mask.png"),
                              os.path.join(defect_gt_dir, f"{stem}.png")):
                if os.path.exists(mask_path):
                    dest_mask = os.path.join(target_train_gt_dir, os.path.basename(mask_path))
                    if not os.path.exists(dest_mask):
                        shutil.copy(mask_path, dest_mask)
                    break
    print(">>> SUP dataset prep done.")
else:
    print(">>> MODE is not 'sup' -> skipping dataset prep.")


>>> SUP prep for 'Nero_SEMISUPERVISED_dustValidationAndTrain', defects: ['grappola', 'paglia', 'nodo_r40', 'nodo', 'splycer']
>>> SUP dataset prep done.


## 3) Grid runner (resume-safe)

Runs each config with identical hyperparameters, saving to its own Drive folder
`ABLATION_BASE/<CATEGORY>__<config>/`. A config that already has a `DONE.txt` is
**skipped**, so you can stop and re-run this cell freely.

To run a subset, set `CONFIG_FILES` manually (e.g. only the single-family ones).

In [ ]:
import os, sys, glob, json, shutil, subprocess

os.chdir(REPO_PATH)


def run_streamed(cmd, log_path=None):
    """Run a command and STREAM its output into the notebook cell.

    subprocess output is NOT shown in Colab by default (it goes to the kernel's
    real stdout, not the cell), which hides tracebacks. We pipe it and re-print
    every line via print(), and optionally tee it to a log file on Drive.
    """
    print(">>>", " ".join(str(c) for c in cmd), flush=True)
    logf = open(log_path, "w", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if logf:
                logf.write(line)
    finally:
        proc.stdout.close()
        rc = proc.wait()
        if logf:
            logf.close()
    return rc


# ===================== CONFIGS TO RUN (edit freely) =====================
# Explicit list, grouped as in configs/EXPERIMENTS.md. Comment out any row you
# don't want, or reorder. To instead run EVERY json present automatically, use
# the glob line at the bottom.
CONFIG_FILES = [
    "configs/aug_off.json",
    "configs/aug_legacy.json",
    "configs/oat_affine.json",
    "configs/oat_blur.json",
    "configs/oat_brightness_contrast.json",
    "configs/oat_dynamic_crop.json",
    "configs/oat_equalize.json",
    "configs/oat_grayscale.json",
    "configs/oat_hflip.json",
    "configs/oat_hue.json",
    "configs/oat_speckle_high.json",
    "configs/oat_speckle_low.json",
    "configs/oat_vflip.json",
    "configs/combined_geometric.json",
    "configs/combined_candidate.json",
]
# Alternative: run every json present, no manual list:
# CONFIG_FILES = sorted(glob.glob('configs/*.json'))

# Keep only the ones that actually exist on disk (guards against typos/renames).
CONFIG_FILES = [c for c in CONFIG_FILES if os.path.exists(c)]
# =======================================================================

RUN_ONNX_EXPORT = True   # also export ONNX (needs the onnx deps from the setup cell)

print(f"{len(CONFIG_FILES)} configs queued:")
for c in CONFIG_FILES:
    print("   -", os.path.basename(c))


def _find_weights(root):
    hits = glob.glob(os.path.join(root, "**", "weights.pt"), recursive=True)
    return sorted(hits)[0] if hits else None


for cfg_path in CONFIG_FILES:
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]
    run_dir  = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}")
    done_marker = os.path.join(run_dir, "DONE.txt")

    if os.path.exists(done_marker):
        print(f"[SKIP] {cfg_name} already completed")
        continue

    os.makedirs(run_dir, exist_ok=True)
    print("\n" + "=" * 64)
    print(f"[RUN ] {cfg_name}  ->  {run_dir}")
    print("=" * 64)

    # --- 1) Train (skip if a checkpoint already exists but the run isn't DONE:
    #        e.g. it crashed during eval/export -> resume at eval, don't retrain). ---
    weights = _find_weights(run_dir)
    if weights:
        print(f"[TRAIN] existing checkpoint found, skipping training -> {weights}")
    else:
        rc = run_streamed([
            "python", "train.py",
            "--dataset", "mvtec",
            "--category", CATEGORY,
            "--mode", MODE,
            "--data_path", DATA_PATH,
            "--datasets_folder", DATA_PATH,
            "--results_save_path", run_dir,
            "--setup_name", f"ssn_{cfg_name}",
            "--num_workers", "1",
            "--backbone", "wide_resnet50_2",
            "--layers", "layer1", "layer2", "layer3",
            "--image_size", "256", "256",
            "--epochs", str(EPOCHS),
            "--batch", str(BATCH),
            "--perlin_thr", "0.34628",
            "--noise_std", "0.10580",
            "--seg_lr", "2.087e-5",
            "--dec_lr", "7.798e-4",
            "--adapt_lr", "0.0001",
            "--patch_size", "5",
            "--gamma", "0.52232",
            "--eval_step_size", "20",
            "--seed", str(SEED),
            "--aug_config", cfg_path,
        ], log_path=os.path.join(run_dir, "train_log.txt"))
        if rc != 0:
            print(f"[FAIL] {cfg_name} training (exit {rc}) -- see train_log.txt above; will retry next run")
            continue
        weights = _find_weights(run_dir)

    if not weights:
        print(f"[WARN] {cfg_name}: training finished but no weights.pt found; skipping eval")
        continue

    # --- 2) Eval: this is what writes the <weights>.calib.json sidecar (train.py
    #        alone does NOT). Without it the ONNX export has no calibration. ---
    print(f"[EVAL] {cfg_name} -> writing .calib.json + eval report")
    ev_rc = run_streamed([
        "python", "eval.py", weights,
        "--dataset", "mvtec",
        "--category", CATEGORY,
        "--datasets_folder", DATA_PATH,
        "--results_save_path", run_dir,
        "--backbone", "wide_resnet50_2",
        "--image_size", "256", "256",
        "--batch", str(BATCH),
        "--num_workers", "1",
        "--seed", str(SEED),
        "--layers", "layer1", "layer2", "layer3",
        "--patch_size", "5",
    ], log_path=os.path.join(run_dir, "eval_log.txt"))
    calib = weights + ".calib.json"
    if ev_rc != 0 or not os.path.exists(calib):
        print(f"[FAIL] {cfg_name} eval/calib (exit {ev_rc}, calib={os.path.exists(calib)}) "
              f"-- see eval_log.txt above; no DONE marker, will retry (training is preserved)")
        continue
    print(f"[calib] OK -> {calib}")

    # --- 3) Optional ONNX export, gathered next to the checkpoint. ---
    if RUN_ONNX_EXPORT:
        print(f"[ONNX] {cfg_name} -> exporting")
        run_streamed(["python", "export_onnx.py", weights],
                     log_path=os.path.join(run_dir, "export_log.txt"))
        onnx_dir = os.path.join(run_dir, "onnx")
        os.makedirs(onnx_dir, exist_ok=True)
        for onnx_file in glob.glob(os.path.join(run_dir, "**", "*.onnx"), recursive=True):
            if os.path.dirname(onnx_file) != onnx_dir:
                shutil.move(onnx_file, os.path.join(onnx_dir, os.path.basename(onnx_file)))
        # keep the calib sidecar alongside the exported model too
        shutil.copy(calib, os.path.join(onnx_dir, os.path.basename(calib)))

    # --- 4) Only now mark the config complete. ---
    shutil.copy(cfg_path, os.path.join(run_dir, "aug_config_used.json"))
    with open(done_marker, "w") as f:
        f.write("completed\n")
    print(f"[DONE] {cfg_name}")

print("\n>>> Grid runner finished this pass.")


15 configs queued:
   - aug_off.json
   - aug_legacy.json
   - oat_affine.json
   - oat_blur.json
   - oat_brightness_contrast.json
   - oat_dynamic_crop.json
   - oat_equalize.json
   - oat_grayscale.json
   - oat_hflip.json
   - oat_hue.json
   - oat_speckle_high.json
   - oat_speckle_low.json
   - oat_vflip.json
   - combined_geometric.json
   - combined_candidate.json

[RUN ] aug_off  ->  /content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation/Nero_SEMISUPERVISED_dustValidationAndTrain__aug_off
>>> python train.py --dataset mvtec --category Nero_SEMISUPERVISED_dustValidationAndTrain --mode sup --data_path /content/mvtec_local --datasets_folder /content/mvtec_local --results_save_path /content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation/Nero_SEMISUPERVISED_dustValidationAndTrain__aug_off --setup_name ssn_aug_off --num_workers 1 --backbone wide_resnet50_2 --layers layer1 layer2 layer3 --image_size 256 256 --epochs 300 --batch 4 --perlin_thr 0.34628 --noise_std 0.

## 4) Summary — collect every config's metrics into one CSV

In [ ]:
import os, glob, json
import pandas as pd

rows = []
for cfg_path in sorted(glob.glob('configs/*.json')):
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]
    run_dir  = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}")
    metric_files = glob.glob(os.path.join(run_dir, "**", "metrics.json"), recursive=True)
    if not metric_files:
        rows.append({"config": cfg_name, "status": "not_run"})
        continue
    with open(sorted(metric_files)[0]) as f:
        m = json.load(f)
    row = {"config": cfg_name, "status": "done"}
    row.update(m)
    rows.append(row)

df = pd.DataFrame(rows)
# Put the most relevant metrics first when present
front = [c for c in ["config", "status", "I-AUROC", "P-AUROC", "AUPRO", "AP-loc",
                     "AP-det", "F1-score"] if c in df.columns]
df = df[front + [c for c in df.columns if c not in front]]

out_csv = os.path.join(ABLATION_BASE, f"ablation_summary_{CATEGORY}.csv")
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print("\nSaved summary ->", out_csv)
